# Training on images and spectra together

This notebook trains a PyTorch model on a multimodal HATS catalog, where every row holds both
an image and a spectrum of the same galaxy. The catalog is a crossmatch of Legacy Survey DR10 images
with DESI EDR spectra, hosted on Hugging Face.

The model is not meant to do real science. It shows how the different column types of a catalog
reach a model:

- **Images** are fixed-shape tensor columns (`tensor[float, (160, 160)]`), one per band.
- **Spectra** are a nested column, with one row of `flux`, `lambda`, `ivar`, ... per object.

Data is streamed with `CatalogDataset`, and several PyTorch workers load batches in parallel while
the model trains. We train once while streaming the catalog from Hugging Face and once from a copy
on local disk, and compare how fast data reaches the GPU in each case.

This notebook is pre-executed because it reads data over the network and is meant for a GPU.

In [1]:
import multiprocessing
import sys
import time
from pathlib import Path

import lsdb
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.nn import functional as F

from lsdb_ml import CatalogDataLoader, CatalogDataset

# Milestone 4: Cross-Modal Data Loader (Month 6; objective 1a).

Definition: A data loader API that allows the training of PyTorch models from multi- modal HATS catalogs; throughput characterization report.

Quantitative Metrics: The team demonstrates training ML models from multiple modalities, and provides an example notebook.

## Introduction 
This notebook trains a PyTorch model on a multimodal HATS catalog, where every row holds both an image and a spectrum of the same galaxy. The catalog is a crossmatch of Legacy Survey DR10 images with DESI EDR spectra, hosted on Hugging Face.

The model used is purely for demonstrating the infrastructural capabilities and is not meant to do real science. For the two modalities:

- **Images** are fixed-shape tensor columns (`tensor[float, (160, 160)]`), one per band.
- **Spectra** are a nested column, with one row of `flux`, `lambda`, `ivar`, ... per object.

This notebook was run on an Nvidia L40, using the dataloader available in lsdb-ml.

Some configuration settings for the notebook are here:

In [2]:
SOURCES = ["remote", "local"] # local writes to LOCAL_PATH if it does not exist
REMOTE_URL = "hf://datasets/brantd/multimodal_hats_example"
LOCAL_PATH = "multimodal_hats_local"
N_PARTITIONS = 32  # of 548, so the notebook runs in minutes
NUM_WORKERS = 4  # PyTorch workers that load batches
EPOCHS = 3

# On macOS, workers that read over HTTPS crash if LSDB was preloaded, so preload only torch there.
PRELOAD = ["torch", "lsdb", "lsdb_ml", "nested_pandas"]
if sys.platform == "darwin" and "remote" in SOURCES:
    PRELOAD = ["torch"]

## Load the catalog

We open only the columns we need, the `g` and `r` band images and the spectra, and keep the first
`N_PARTITIONS` partitions. For `"local"`, the same partitions are written to disk and opened from
there.

In [3]:
columns = ["image_flux_g", "image_flux_r", "spectrum"]
catalogs = {"remote": lsdb.open_catalog(REMOTE_URL, columns=columns).partitions[:N_PARTITIONS]}
if "local" in SOURCES:
    # collection.properties is written last, so it exists only if a previous write finished.
    if not Path(LOCAL_PATH, "collection.properties").exists():
        catalogs["remote"].write_catalog(LOCAL_PATH, overwrite=True)
    catalogs["local"] = lsdb.open_catalog(LOCAL_PATH, columns=columns)
catalogs["remote"]

,image_flux_g,image_flux_r,ra_mmu_legacysurvey_dr10_south_21,dec_mmu_legacysurvey_dr10_south_21,spectrum
npartitions=32,,,,,
"Order: 7, Pixel: 41472","tensor[float, (160, 160)]","tensor[float, (160, 160)]",double[pyarrow],double[pyarrow],"nested<flux: [float], ivar: [float], lsf_sigma..."
...,...,...,...,...,...
"Order: 7, Pixel: 41602",...,...,...,...,...
"Order: 7, Pixel: 41603",...,...,...,...,...


## Turn rows into tensors

The default `collate_catalog` handles flat and nested numeric columns, but not tensor columns, so
we write a `collate_fn`. It receives a `NestedFrame` with one batch of rows and returns tensors:

- `to_stack()` turns a tensor column into one `(batch, 160, 160)` array. We stack the two bands
  into channels.
- Every DESI spectrum is sampled on the same wavelength grid, so the flat `flux` values reshape into
  a `(batch, n_wavelengths)` array.

In [4]:
def collate(frame):
    image = np.stack([frame["image_flux_g"].to_stack(), frame["image_flux_r"].to_stack()], axis=1)
    spectrum = frame["spectrum"].nest["flux"].to_numpy(dtype=np.float32).reshape(len(frame), -1)
    return {"image": torch.from_numpy(image), "spectrum": torch.tensor(spectrum)}


example = collate(catalogs[SOURCES[0]].partitions[0].compute())
{name: tensor.shape for name, tensor in example.items()}

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

{'image': torch.Size([7, 2, 160, 160]), 'spectrum': torch.Size([7, 7781])}

## Stream batches with several workers

With `num_workers > 0`, each PyTorch worker streams its own share of the partitions with dask's
synchronous scheduler, so no dask client is needed.

In [5]:
context = multiprocessing.get_context("forkserver")
context.set_forkserver_preload(PRELOAD) # preload packages for workers


def make_loader(catalog):
    dataset = CatalogDataset(catalog, batch_size=64, seed=0, drop_last=True, collate_fn=collate)
    return CatalogDataLoader(dataset, num_workers=NUM_WORKERS, multiprocessing_context=context)

## A two-tower model

Each modality gets its own encoder. The images go through a standard ResNet-18 from torchvision,
with its first layer changed to take 2 bands instead of 3 colors. The spectra go through a small 1D
CNN. Both map to a shared 64-dimensional embedding. A `BatchNorm` at the input of each encoder takes
care of the very different flux scales of the two.

We train it contrastively, as in CLIP or AstroCLIP: in a batch of `B` objects, the image of each
object should be closer to its own spectrum than to the other `B - 1` spectra. No labels are
needed.

In [6]:
class TwoTower(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        resnet = torchvision.models.resnet18(num_classes=dim)
        resnet.conv1 = nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False)  # 2 bands, not RGB
        self.image = nn.Sequential(nn.BatchNorm2d(2), resnet)
        self.spectrum = nn.Sequential(
            nn.BatchNorm1d(1),
            nn.Conv1d(1, 16, 9, stride=8),
            nn.ReLU(),
            nn.Conv1d(16, 32, 9, stride=8),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, dim),
        )

    def forward(self, image, spectrum):
        return F.normalize(self.image(image), dim=1), F.normalize(self.spectrum(spectrum[:, None]), dim=1)

## Train

Each source trains a fresh model for `EPOCHS` epochs, so the runs differ only in where the data
comes from. `CatalogDataLoader` gives each epoch a new shuffle order.

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)


def new_model():
    torch.manual_seed(0)
    model = TwoTower().to(device)
    return model, torch.optim.Adam(model.parameters(), lr=1e-3)


def train_step(model, optimizer, batch):
    image_emb, spectrum_emb = model(batch["image"].to(device), batch["spectrum"].to(device))
    logits = image_emb @ spectrum_emb.T / 0.1
    target = torch.arange(len(logits), device=device)
    loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # .item() waits for the GPU, so timing a call to train_step includes its GPU work.
    return loss.item(), (logits.argmax(1) == target).float().mean().item()


def train(source):
    """Train a fresh model on one source; returns one row of timings per batch."""
    model, optimizer = new_model()
    loader = make_loader(catalogs[source])
    log = []
    for epoch in range(EPOCHS):
        last = time.perf_counter()
        for batch in loader:
            loaded = time.perf_counter()
            loss, matched = train_step(model, optimizer, batch)
            done = time.perf_counter()
            log.append(
                dict(
                    source=source,
                    epoch=epoch,
                    wait_s=loaded - last,
                    step_s=done - loaded,
                    loss=loss,
                    matched=matched,
                )
            )
            last = done
        epoch_log = pd.DataFrame(log).query("epoch == @epoch")
        print(
            f"{source}, epoch {epoch}: loss {epoch_log.loss.mean():.3f}, matched {epoch_log.matched.mean():.1%}"
        )
    return pd.DataFrame(log)


log = pd.concat([train(source) for source in SOURCES], ignore_index=True)

/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/site-packages/lance/__init__.py:358: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn or forkserver instead.
  warnings.warn(
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer al

remote, epoch 0: loss 3.291, matched 9.1%


/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been rese

remote, epoch 1: loss 3.013, matched 11.4%


/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been rese

remote, epoch 2: loss 2.914, matched 12.5%


... (2 outputs left out)

/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been rese

local, epoch 1: loss 3.014, matched 11.8%


/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid = os.fork()
/astro/users/brantd/.conda/envs/aion_training/lib/python3.13/multiprocessing/forkserver.py:287: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been rese

local, epoch 2: loss 2.925, matched 12.1%


## Throughput report

Is the GPU waiting on LSDB, or LSDB on the GPU? Three numbers tell us:

1. **Data rate.** Objects per second that the loader delivered during training, and the size of
   those tensors in MB per second. (This is the in-memory size, not the compressed bytes read.)
2. **GPU ceiling.** Objects per second the training step can process when the data is already in
   memory, measured by repeating the step on one batch. It does not depend on the source.
3. **GPU-busy fraction.** The share of wall time the loop spent in the training step. The rest was
   spent waiting for the next batch.

If the GPU-busy fraction is close to 1, the model is the bottleneck and faster loading would not
help. If it is low, the GPU sits idle and training runs only as fast as the data arrives.

The step time includes copying the batch to the device and some Python overhead, so the busy
fraction is an upper bound on how busy the GPU itself was.

In [8]:
# One batch, loaded in this process without workers.
batch = next(iter(CatalogDataset(catalogs[SOURCES[-1]], batch_size=64, collate_fn=collate)))
model, optimizer = new_model()
train_step(model, optimizer, batch)  # warm up
start = time.perf_counter()
for _ in range(20):
    train_step(model, optimizer, batch)
gpu_ceiling = 20 * len(batch["image"]) / (time.perf_counter() - start)
print(f"GPU ceiling: {gpu_ceiling:,.0f} objects/s on {device}")

GPU ceiling: 943 objects/s on cuda


Each epoch's numbers come from the timings logged during training, split in two parts:

- **Startup.** `startup_s` is the wait for the epoch's first batch, while the workers start and read
  their first partitions. `epoch_s` is the whole epoch, including it.
- **Steady state.** `objects_per_s`, `MB_per_s`, `gpu_busy` and `gpu_headroom` cover the rest of the
  epoch, once the workers are running. `gpu_headroom` is how many times faster the GPU could train
  than the data arrived.

In [9]:
mb_per_object = sum(tensor[0].nbytes for tensor in batch.values()) / 1e6
report = log.groupby(["source", "epoch"], sort=False).agg(
    batches=("step_s", "size"),
    startup_s=("wait_s", "first"),
    wait_s=("wait_s", "sum"),
    step_s=("step_s", "sum"),
)
report["epoch_s"] = report.wait_s + report.step_s
steady_s = report.epoch_s - report.startup_s  # time after the first batch arrived
report["objects_per_s"] = report.batches * len(batch["image"]) / steady_s
report["MB_per_s"] = report.objects_per_s * mb_per_object
report["gpu_busy"] = report.step_s / steady_s
report["gpu_headroom"] = gpu_ceiling / report.objects_per_s
print(f"{NUM_WORKERS} workers, preloaded {PRELOAD}")
report[["batches", "epoch_s", "startup_s", "objects_per_s", "MB_per_s", "gpu_busy", "gpu_headroom"]].round(2)

4 workers, preloaded ['torch', 'lsdb', 'lsdb_ml', 'nested_pandas']


batches  epoch_s  startup_s  objects_per_s  MB_per_s  gpu_busy  \
source epoch                                                                   
remote 0          164   156.08      26.11          80.75     19.05      0.09   
       1          163    59.15      12.01         221.30     52.21      0.23   
       2          164    88.23       7.25         129.61     30.58      0.13   
local  0          164    12.15       1.13         951.90    224.58      0.98   
       1          163    12.18       1.11         941.89    222.21      0.97   
       2          164    12.18       1.06         944.12    222.74      0.97   

              gpu_headroom  
source epoch                
remote 0             11.67  
       1              4.26  
       2              7.27  
local  0              0.99  
       1              1.00  
       2              1.00

### Reading the results

The `gpu_busy` column says which side is the bottleneck:

- **Close to 1: the model is the bottleneck.** Faster loading would not help; a larger batch, mixed
  precision or a faster GPU would.
- **Well below 1: the data is the bottleneck.** The GPU sits idle between batches, and training runs
  only as fast as `objects_per_s`. `gpu_headroom` says how much faster it could go.

**Remote vs local.** The two runs use the same partitions, model and workers, so the difference
between them is what reading over the network costs. Each partition holds about 100 MB of images
and spectra. Reading from Hugging Face, the data rate is set by how fast those can be downloaded,
and the GPU waits on data. Reading the local copy removes that limit, and the balance then depends
on the model.

**Worker startup** is a fixed cost per epoch, so compare `startup_s` with `epoch_s`. A worker's
first batch waits until its first partition has been read and converted, and with short epochs this
can take a large share of each epoch even while the steady state keeps the GPU busy.
`CatalogDataLoader(..., persistent_workers=True)` keeps the workers between epochs, but then every
epoch repeats the same shuffle order.